In [1]:
!pip install /kaggle/input/pdmupf/pymupdf-1.26.4-cp39-abi3-manylinux_2_28_x86_64.whl

Processing /kaggle/input/pdmupf/pymupdf-1.26.4-cp39-abi3-manylinux_2_28_x86_64.whl


In [2]:
from nltk.tokenize import sent_tokenize, PunktSentenceTokenizer
import pandas as pd
import re
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
from pathlib import Path
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import pymupdf
import logging
from tqdm import tqdm
import numpy as np
import uuid

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DatasetMentionPipeline:
    def __init__(self, ner_model_path, classifier_model_path):
        """Initialize the pipeline with NER and classifier models"""
        logger.info("Loading NER model...")
        self.ner_tokenizer = AutoTokenizer.from_pretrained(ner_model_path)
        self.ner_model = AutoModelForTokenClassification.from_pretrained(ner_model_path)
        
        logger.info("Loading classifier model...")
        self.classifier_tokenizer = AutoTokenizer.from_pretrained(classifier_model_path)
        self.classifier_model = AutoModelForSequenceClassification.from_pretrained(classifier_model_path)
        
        # Set device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Using device: {self.device}")
        
        # Move models to device
        self.ner_model.to(self.device)
        self.classifier_model.to(self.device)
        self.sentence_tokenizer = PunktSentenceTokenizer()
        
        # Set models to evaluation mode
        self.ner_model.eval()
        self.classifier_model.eval()
        
        logger.info("Models loaded successfully")

    def is_header_footer(self, text: str, page_num: int, total_pages: int) -> bool:
        """Enhanced header/footer detection"""
        patterns = [
            r"^\s*\d{1,3}\s*$",  # Page numbers
            r"^.*\|\s*\d{1,3}\s*of\s*\d{1,3}\s*\|.*$",  # Page X of Y
            r"^.*\d{1,2}\s*[–-]\s*\d{1,2}\s*[a-zA-Z]+\s*\d{4}.*$",  # Date patterns
            r"^.*©.*$",  # Copyright symbols
            r"^.*confidential.*$",  # Confidential notices
            r"^.*draft.*$",  # Draft watermarks
            r"^.*(this\s+copy\s+is\s+for\s+review).*$",  # Review copies
        ]
        
        for pattern in patterns:
            if re.match(pattern, text, re.IGNORECASE):
                return True
        
        # Special handling for first and last pages
        if page_num == 1 and ("abstract" in text.lower() or "introduction" in text.lower()):
            return False
        if page_num == total_pages and ("references" in text.lower() or "acknowledg" in text.lower()):
            return False
            
        return False

    def extract_text_from_pdf(self, pdf_file):
        """Enhanced PDF text extraction with header/footer filtering"""
        full_text = []
        try:
            with pymupdf.open(pdf_file) as doc:
                total_pages = len(doc)
                for page_num, page in enumerate(doc, start=1):
                    blocks = page.get_text("blocks", sort=True)
                    page_text = [
                        re.sub(r'\s+', ' ', block[4]).strip()
                        for block in blocks
                        if block[6] == 0 and not self.is_header_footer(block[4], page_num, total_pages)
                    ]
                    if page_text:
                        full_text.append("\n".join(page_text))
            
            result = "\n\n".join(full_text)
            logger.info(f"Extracted {len(result)} characters from PDF {pdf_file.name}")
            return result
        except Exception as e:
            logger.error(f"Error extracting text from PDF {pdf_file.name}: {str(e)}")
            return ""

    def extract_text_from_xml(self, xml_file):
        """Enhanced XML text extraction with multiple fallback strategies"""
        try:
            with open(xml_file, 'r', encoding='utf-8', errors='replace') as f:
                content = f.read()

            # Parse with BeautifulSoup first
            try:
                soup = BeautifulSoup(content, 'lxml-xml')
                for tag in soup(['ref-list', 'bibliography', 'table-wrap', 'fig']):
                    tag.decompose()
                text = soup.get_text(separator='\n', strip=True)
            except Exception as e:
                logger.warning(f"BeautifulSoup failed for {xml_file.name}, trying ElementTree: {e}")
                text = ""

            # Fallback to ElementTree if BeautifulSoup didn't work
            if not text:
                try:
                    root = ET.fromstring(content)
                    text = "\n".join([
                        el.text.strip() for el in root.findall('.//')
                        if el.text and el.text.strip()
                    ])
                except Exception as e:
                    logger.warning(f"ElementTree failed for {xml_file.name}, using raw tag removal: {e}")
                    text = re.sub(r'<[^>]+>', '', content)  # Raw tag removal

            # Post-processing
            text = re.sub(r'\s+', ' ', text)
            text = re.sub(r'\n{3,}', '\n\n', text)
            text = re.sub(r'\b(doi|DOI):\s*(\S+)', r'https://doi.org/\2', text)  # Enhanced DOI handling
            
            logger.info(f"Extracted {len(text)} characters from XML {xml_file.name}")
            return text.strip()
        except Exception as e:
            logger.error(f"Error extracting text from XML {xml_file.name}: {str(e)}")
            return ""

    def process_document(self, file_path):
        """Process a single document and extract text based on file extension"""
        file_ext = file_path.suffix.lower()
        
        if file_ext == '.pdf':
            return self.extract_text_from_pdf(file_path)
        elif file_ext in ['.xml', '.nxml']:
            return self.extract_text_from_xml(file_path)
        else:
            logger.warning(f"Unsupported file type: {file_path.name}")
            return ""

    def extract_mentions(self, text):
        """Extract dataset mentions using NER model with sliding window approach"""
        if not text:
            return []
        
        # Use the same parameters as during training
        MAX_LENGTH = 512
        STRIDE = 256
        CHUNK_SIZE = MAX_LENGTH - 2  # Account for [CLS] and [SEP]
        
        # Tokenize the entire text
        encoding = self.ner_tokenizer(
            text,
            return_offsets_mapping=True,
            add_special_tokens=False,
            truncation=False
        )
        
        tokens = encoding["input_ids"]
        offset_mapping = encoding["offset_mapping"]
        
        # Create sliding window chunks
        chunks = []
        start = 0
        
        while start < len(tokens):
            end = min(start + CHUNK_SIZE, len(tokens))
            
            # Extract chunk tokens
            chunk_tokens = tokens[start:end]
            
            # Add special tokens
            input_ids = [self.ner_tokenizer.cls_token_id] + chunk_tokens + [self.ner_tokenizer.sep_token_id]
            
            # Create attention mask
            attention_mask = [1] * len(input_ids)
            
            chunks.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "token_offset": start,  # Track original position
                "token_length": len(chunk_tokens)  # Number of tokens in chunk (without special tokens)
            })
            
            # Exit if at end of text
            if end == len(tokens):
                break
                
            # Apply stride (with overlap)
            start += (CHUNK_SIZE - STRIDE)
        
        # Process each chunk
        all_mentions = []
        
        for chunk in chunks:
            try:
                # Prepare inputs
                input_ids = torch.tensor([chunk["input_ids"]]).to(self.device)
                attention_mask = torch.tensor([chunk["attention_mask"]]).to(self.device)
                
                # Predict
                with torch.no_grad():
                    outputs = self.ner_model(input_ids=input_ids, attention_mask=attention_mask)
                
                # Process predictions
                predictions = torch.argmax(outputs.logits, dim=2)
                predicted_labels = [self.ner_model.config.id2label[p.item()] for p in predictions[0]]
                
                # Skip special tokens ([CLS] and [SEP])
                token_labels = predicted_labels[1:1+chunk["token_length"]]
                
                # Extract mentions from this chunk
                current_mention_tokens = []
                
                for i, label in enumerate(token_labels):
                    token_idx = chunk["token_offset"] + i
                    
                    if label == "B-DATASET":
                        # Save previous mention if exists
                        if current_mention_tokens:
                            mention = self._extract_mention_from_token_indices(text, offset_mapping, current_mention_tokens)
                            if mention and len(mention.strip()) > 2:
                                all_mentions.append(mention.strip())
                        
                        # Start new mention
                        current_mention_tokens = [token_idx]
                        
                    elif label == "I-DATASET" and current_mention_tokens:
                        current_mention_tokens.append(token_idx)
                        
                    elif current_mention_tokens:
                        # End of mention
                        mention = self._extract_mention_from_token_indices(text, offset_mapping, current_mention_tokens)
                        if mention and len(mention.strip()) > 2:
                            all_mentions.append(mention.strip())
                        current_mention_tokens = []
                
                # Handle mention at end of chunk
                if current_mention_tokens:
                    mention = self._extract_mention_from_token_indices(text, offset_mapping, current_mention_tokens)
                    if mention and len(mention.strip()) > 2:
                        all_mentions.append(mention.strip())
                        
            except Exception as e:
                logger.error(f"Error processing chunk: {str(e)}")
                continue
        
        # Remove duplicates while preserving order
        unique_mentions = []
        seen = set()
        for mention in all_mentions:
            mention_clean = mention.lower().strip()
            if mention_clean not in seen and len(mention_clean) > 2:
                seen.add(mention_clean)
                unique_mentions.append(mention)
        
        logger.info(f"Found {len(unique_mentions)} unique mentions")
        return unique_mentions

    def _extract_mention_from_token_indices(self, text, offset_mapping, token_indices):
        """Extract mention text using token indices and offset mapping"""
        if not token_indices:
            return ""
        
        # Get the character offsets for the first and last tokens
        start_char = offset_mapping[token_indices[0]][0]
        end_char = offset_mapping[token_indices[-1]][1]
        
        # Extract the text
        mention = text[start_char:end_char]
        
        # Clean up the mention
        mention = re.sub(r'\s+', ' ', mention).strip()
        return mention

    def extract_context(self, text, mention_text, method='sentence', n_sentences=1, window_size=150):
        """
        Extract context around a mention using the same approach as during training
        
        Args:
            text: Full text of the document
            mention_text: The extracted mention text
            method: 'char' for character window or 'sentence' for sentence-based window
            n_sentences: Number of sentences to include on each side (for 'sentence' method)
            window_size: Number of characters to include on each side (for 'char' method)
        """
        if not text or not mention_text:
            return mention_text  # Return the mention itself as fallback
        
        # Try multiple matching strategies to find the mention in text
        patterns_to_try = [
            re.escape(mention_text),  # Exact match
            re.escape(mention_text.lower()),  # Case insensitive
            r'\b' + re.escape(mention_text) + r'\b',  # Word boundaries
        ]
        
        match = None
        for pattern in patterns_to_try:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                break
        
        if not match:
            # Fallback: find the longest common substring
            text_lower = text.lower()
            mention_lower = mention_text.lower()
            
            best_pos = -1
            best_len = 0
            
            for i in range(len(text_lower) - len(mention_lower) + 1):
                if text_lower[i:i+len(mention_lower)] == mention_lower:
                    best_pos = i
                    best_len = len(mention_lower)
                    break
            
            if best_pos == -1:
                return mention_text  # Return the mention itself as context
            
            start_idx, end_idx = best_pos, best_pos + best_len
        else:
            start_idx, end_idx = match.span()
        
        # Extract context based on the specified method
        if method == 'sentence':
            try:
                # Get sentence spans
                sentence_spans = list(self.sentence_tokenizer.span_tokenize(text))
                
                # Find containing sentence
                sent_idx = None
                for i, (s_start, s_end) in enumerate(sentence_spans):
                    if s_start <= start_idx < s_end:
                        sent_idx = i
                        break
                
                if sent_idx is not None:
                    # Get context window
                    start_sent_idx = max(0, sent_idx - n_sentences)
                    end_sent_idx = min(len(sentence_spans), sent_idx + n_sentences + 1)
                    context_start = sentence_spans[start_sent_idx][0]
                    context_end = sentence_spans[end_sent_idx-1][1]
                    return text[context_start:context_end].strip()
                else:
                    # Fallback to character window if sentence not found
                    context_start = max(0, start_idx - window_size)
                    context_end = min(len(text), end_idx + window_size)
                    return text[context_start:context_end].strip()
                    
            except Exception as e:
                logger.warning(f"Sentence tokenization failed: {e}, falling back to character window")
                # Fallback to character window
                context_start = max(0, start_idx - window_size)
                context_end = min(len(text), end_idx + window_size)
                return text[context_start:context_end].strip()
                
        else:  # Character-based method
            context_start = max(0, start_idx - window_size)
            context_end = min(len(text), end_idx + window_size)
            return text[context_start:context_end].strip()

    def classify_mention(self, context_text):
        """Classify mention as Primary or Secondary with improved handling"""
        if not context_text.strip():
            return "Secondary"  # Default if no context
        
        try:
        # Tokenize with the same settings as during training
            inputs = self.classifier_tokenizer(   
               context_text,
               padding="max_length",
               truncation=True,
               max_length=512,
               return_tensors="pt"
        )
            
            # Move to device
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # Predict
            with torch.no_grad():
                outputs = self.classifier_model(**inputs)
            
            # Get prediction with confidence
            probabilities = torch.softmax(outputs.logits, dim=1)
            predicted_class_id = torch.argmax(probabilities, dim=1).item()
            confidence = probabilities[0][predicted_class_id].item()
            
            # Map to label (assuming 0=Primary, 1=Secondary based on common setup)
            if predicted_class_id == 0:
                result = "Primary"
            else:
                result = "Secondary"
            
            logger.debug(f"Classification: {result} (confidence: {confidence:.3f})")
            return result
            
        except Exception as e:
            logger.error(f"Error classifying mention: {str(e)}")
            return "Secondary"  # Default on error
  
    def extract_dataset_id(self, mention_text):
        """Extract dataset ID from mention text with focus on DOIs and valid accession numbers"""
        if not mention_text:
            return None

        # Clean the mention text
        mention_clean = re.sub(r'\s+', ' ', mention_text.strip())
        
        # Enhanced DOI patterns to capture complete DOIs in various formats
        doi_patterns = [
            # Standard DOI URLs
            r'https?://doi\.org/(10\.\d{4,9}/[-._;()/:A-Z0-9]+)',
            r'https?://dx\.doi\.org/(10\.\d{4,9}/[-._;()/:A-Z0-9]+)',
            
            # DOI with domain but without protocol
            r'doi\.org/(10\.\d{4,9}/[-._;()/:A-Z0-9]+)',
            r'dx\.doi\.org/(10\.\d{4,9}/[-._;()/:A-Z0-9]+)',
            
            # DOIs in publisher URLs
            r'https?://[a-z0-9.-]+/doi/(10\.\d{4,9}/[-._;()/:A-Z0-9]+)',
            r'https?://[a-z0-9.-]+/doi/(10\.\d{4,9}/[-._;()/:A-Z0-9]+)',
            
            # Raw DOI pattern (as fallback)
            r'\b(10\.\d{4,9}/[-._;()/:A-Z0-9]+)\b',
        ]
        
        for pattern in doi_patterns:
            match = re.search(pattern, mention_clean, re.IGNORECASE)
            if match:
                doi = match.group(1)
                if re.match(r'10\.\d{4,9}/.+', doi):
                    return f"https://doi.org/{doi}"

        # Patterns for non-DOI repositories (raw accession numbers)
        accession_patterns = [
            # GEO
            (r'\b(GSE\d+)\b', lambda x: x.group(1)),
            (r'\b(GSM\d+)\b', lambda x: x.group(1)),
            (r'\b(GDS\d+)\b', lambda x: x.group(1)),
            (r'\b(GPL\d+)\b', lambda x: x.group(1)),
            # SRA
            (r'\b(SRR\d+)\b', lambda x: x.group(1)),
            (r'\b(ERR\d+)\b', lambda x: x.group(1)),
            (r'\b(DRR\d+)\b', lambda x: x.group(1)),
            # ChemBL
            (r'\b(CHEMBL\d+)\b', lambda x: x.group(1)),
            # Other common identifiers
            (r'\b(SAM[NED]\d+)\b', lambda x: x.group(1)),
            (r'\b(EPI_ISL_\d+)\b', lambda x: x.group(1)),
            (r'\b(ENS[GT]\d+)\b', lambda x: x.group(1)),
            (r'\b(IPR\d+)\b', lambda x: x.group(1)),
            (r'\b(PF\d+)\b', lambda x: x.group(1)),
            (r'\b(rs\d+)\b', lambda x: x.group(1)),
            (r'\b(CVCL_\d+)\b', lambda x: x.group(1)),
            (r'\b(E-[A-Z]+-\d+)\b', lambda x: x.group(1)),
            (r'\b(K\d+)\b', lambda x: x.group(1)),
        ]
        
        for pattern, formatter in accession_patterns:
            match = re.search(pattern, mention_clean, re.IGNORECASE)
            if match:
                try:
                    result = formatter(match)
                    if result and self.is_valid_dataset_id(result):
                        return result
                except:
                    continue

        # Fallback patterns for other potential IDs (more restrictive)
        fallback_patterns = [
            r'\b([A-Z]{3,}\d{3,}[A-Z0-9]*)\b',  # UPPERCASE letters followed by numbers
            r'\b(\d{5,}[A-Z]+[A-Z0-9]*)\b',  # Numbers followed by UPPERCASE letters
        ]
        
        for pattern in fallback_patterns:
            match = re.search(pattern, mention_clean)
            if match:
                candidate = match.group(1)
                if self.is_valid_dataset_id(candidate):
                    return candidate
                    
        return None

    def is_valid_dataset_id(self, dataset_id):
        """Strict validation for dataset IDs"""
        if not dataset_id or not isinstance(dataset_id, str):
            return False
            
        clean_id = dataset_id.strip()
        if len(clean_id) < 6:
            return False
            
        # List of invalid strings
        invalid_matches = [
            'https', 'http', 'doi', 'dryad', 'zenodo', 'figshare', 
            'ccdc', 'pangaea', 'usgs', 'www', 'org', 'com', 'net',
            'chem', 'primary', 'secondary', 'doi.org', 'dx.doi.org',
            'example', 'test', 'sample'
        ]
        
        if clean_id.lower() in invalid_matches:
            return False
            
        # Check for domain-like patterns
        if re.match(r'^[a-z0-9]+\.[a-z]{2,}$', clean_id, re.IGNORECASE) or \
           re.match(r'^[a-z0-9]+\.[a-z0-9]+\.[a-z]+$', clean_id, re.IGNORECASE):
            return False
            
        # Check for year patterns
        if re.match(r'^(19|20)\d{2}$', clean_id):
            return False
            
        # Check for numeric only
        if clean_id.isdigit():
            if len(clean_id) < 6 or int(clean_id) < 1000:
                return False
            
        # Check for alphabetic only
        if clean_id.isalpha() and len(clean_id) < 4:
            return False
            
        # Invalid patterns
        invalid_patterns = [
            r'^(fig|figure|tab|table|eq|equation|sect|section|ref|reference)s?$',
            r'^(the|and|or|of|in|to|for|with|by|from|up|on|at|is|are|was|were)$',
            r'^[a-z]{1,3}$',
            r'^\W+$',
            r'^https?://',
            r'^www\.',
            r'^doi\.',
            r'^dx\.',
        ]
        
        for pattern in invalid_patterns:
            if re.match(pattern, clean_id, re.IGNORECASE):
                return False
                
        return True

    def normalize_id(self, dataset_id):
        """Normalize dataset IDs to canonical form"""
        if not dataset_id:
            return None
            
        cleaned = dataset_id.strip()
        
        # If already a DOI URL, return as is
        if cleaned.startswith('https://doi.org/') or cleaned.startswith('http://doi.org/'):
            return cleaned
            
        # If it contains a DOI pattern, convert to DOI URL
        if 'doi' in cleaned.lower():
            doi_match = re.search(r'10\.\d{4,9}/[^\s\]\)";,]+', cleaned)
            if doi_match:
                return f"https://doi.org/{doi_match.group(0)}"
                
        # For non-DOI IDs, return the raw ID after basic cleaning
        cleaned = re.sub(r'\s+', '', cleaned)
        cleaned = re.sub(r'[^\w.\-]', '', cleaned)
        
        return cleaned if cleaned and self.is_valid_dataset_id(cleaned) else None


    def process_test_data(self, xml_dir, pdf_dir, output_file):
        """Process all test files and generate predictions with improved handling"""
        xml_dir = Path(xml_dir)
        pdf_dir = Path(pdf_dir)
        results = []

        # Get all test files
        xml_files = list(xml_dir.glob("*.xml")) + list(xml_dir.glob("*.nxml"))
        pdf_files = list(pdf_dir.glob("*.pdf"))
        test_files = xml_files + pdf_files
        
        logger.info(f"Found {len(xml_files)} XML files and {len(pdf_files)} PDF files")

        stats = {
            'files_processed': 0,
            'files_with_text': 0,
            'files_with_mentions': 0,
            'total_mentions': 0,
            'mentions_with_valid_id': 0,
            'primary_mentions': 0,
            'secondary_mentions': 0,
            'files_without_text': [],
            'files_without_mentions': []
        }

        for file_path in tqdm(test_files, desc="Processing files"):
            article_id = file_path.stem
            stats['files_processed'] += 1

            # Extract text
            text = self.process_document(file_path)
            if not text.strip():
                stats['files_without_text'].append(article_id)
                logger.warning(f"No text extracted from {file_path.name}")
                continue

            stats['files_with_text'] += 1
            
            # Extract mentions
            mentions = self.extract_mentions(text)
            if not mentions:
                stats['files_without_mentions'].append(article_id)
                logger.warning(f"No mentions found in {file_path.name}")
                continue

            stats['files_with_mentions'] += 1
            stats['total_mentions'] += len(mentions)
            
            logger.info(f"Processing {article_id}: found {len(mentions)} mentions")

            # Process each mention
            processed_ids = set()  # Track processed IDs for deduplication
            
            for mention_text in mentions:
                # Extract dataset ID
                dataset_id = self.extract_dataset_id(mention_text)
                if not dataset_id:
                    logger.debug(f"No valid ID extracted from: {mention_text}")
                    continue

                # Normalize ID
                dataset_id = self.normalize_id(dataset_id)
                if not dataset_id or not self.is_valid_dataset_id(dataset_id):
                    logger.debug(f"Invalid ID after normalization: {dataset_id}")
                    continue

                # Check for duplicates
                if (article_id, dataset_id) in processed_ids:
                    continue
                processed_ids.add((article_id, dataset_id))

                # Extract context and classify
                context = self.extract_context(text, mention_text, method='sentence', n_sentences=1)
                mention_type = self.classify_mention(context)

                stats['mentions_with_valid_id'] += 1
                if mention_type == "Primary":
                    stats['primary_mentions'] += 1
                else:
                    stats['secondary_mentions'] += 1

                results.append({
                    'article_id': article_id,
                    'dataset_id': dataset_id,
                    'type': mention_type,
                    'mention_text': mention_text  # Keep for debugging
                })

        # Log statistics
        logger.info("\n=== Processing Summary ===")
        logger.info(f"Total files processed: {stats['files_processed']}")
        logger.info(f"Files with text: {stats['files_with_text']}")
        logger.info(f"Files with mentions: {stats['files_with_mentions']}")
        logger.info(f"Total mentions found: {stats['total_mentions']}")
        logger.info(f"Mentions with valid IDs: {stats['mentions_with_valid_id']}")
        logger.info(f"Primary mentions: {stats['primary_mentions']}")
        logger.info(f"Secondary mentions: {stats['secondary_mentions']}")

        if stats['files_without_text']:
            logger.warning(f"Files without text ({len(stats['files_without_text'])}): {stats['files_without_text'][:5]}...")
        if stats['files_without_mentions']:
            logger.warning(f"Files without mentions ({len(stats['files_without_mentions'])}): {stats['files_without_mentions'][:5]}...")

        # Create results DataFrame
        if results:
            results_df = pd.DataFrame(results)
            # Remove duplicates (keep first occurrence)
            initial_count = len(results_df)
            results_df = results_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first")
            final_count = len(results_df)
            
            if initial_count != final_count:
                logger.info(f"Removed {initial_count - final_count} duplicate mentions")
            
            # Prepare final output
            results_df.insert(0, "row_id", range(len(results_df)))
            output_df = results_df[['row_id', 'article_id', 'dataset_id', 'type']]
            
            # Save results
            output_df.to_csv(output_file, index=False)
            logger.info(f"Saved {len(output_df)} predictions to {output_file}")
            
            # Show sample results
            logger.info("\n=== Sample Predictions ===")
            for i, row in output_df.head(10).iterrows():
                logger.info(f"{row['article_id']}: {row['dataset_id']} ({row['type']})")
                
        else:
            logger.warning("No valid predictions generated!")
            # Create empty DataFrame with correct structure
            output_df = pd.DataFrame(columns=['row_id', 'article_id', 'dataset_id', 'type'])
            output_df.to_csv(output_file, index=False)

        return output_df, stats

# Main execution
if __name__ == "__main__":
    # Initialize pipeline
    ner_model_path = "/kaggle/input/ner_ultimate_3/pytorch/default/1/ner_model_final_ultimate_3"
    classifier_model_path = "/kaggle/input/data_citation_classification_model/pytorch/default/1/final_data_citation_classification_model"
    
    pipeline = DatasetMentionPipeline(ner_model_path, classifier_model_path)
    
    # Process test data
    base = Path("/kaggle/input/make-data-count-finding-data-references")
    base1 = Path("/kaggle/working")
    xml_dir = base / "test/XML"
    pdf_dir = base / "test/PDF"
    output_file = base1 / "submission.csv"
    
    results, stats = pipeline.process_test_data(xml_dir, pdf_dir, output_file)
    
    print("\n" + "="*50)
    print("FINAL SUMMARY")
    print("="*50)
    print(f"Total predictions: {len(results)}")
    if len(results) > 0:
        print(f"Primary/Secondary ratio: {stats['primary_mentions']}/{stats['secondary_mentions']}")
        print(f"Average mentions per document: {stats['total_mentions']/max(1, stats['files_with_mentions']):.1f}")
        print("\nFirst 5 predictions:")
        print(results.head())
    else:
        print("⚠️  WARNING: No predictions generated!")
        print("This would result in a score of 0.")
        print("Check the logs above for specific issues.")


2025-08-31 06:59:44.712161: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756623584.876192      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756623584.927833      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Processing files: 100%|██████████| 55/55 [01:17<00:00,  1.41s/it]


FINAL SUMMARY
Total predictions: 0
⚠️  WARNING: No predictions generated!
This would result in a score of 0.
Check the logs above for specific issues.
